# Assignment 3: Fine-tuning language models

In this assignment, you will perform supervised fine-tuning (SFT) of a small open LLM on an instruction tuning dataset. You will convert this dataset into instruction-response pairs, fine-tune a causal language model using LoRA (Low-Rank Adaptation), and evaluate it through prompted inference and comparison with other methods.

## Preliminaries

First, let's install the required libraries. If you are running in your own environment, make sure the following are installed:

- [Torch](https://docs.pytorch.org/docs/stable/index.html)
- [Transformers](https://huggingface.co/docs/transformers/index)
- [Datasets](https://huggingface.co/docs/datasets/index)
- [Evaluate](https://huggingface.co/docs/evaluate/en/index)
- [NLTK](https://www.nltk.org/api/nltk.html)
- [rouge_score](https://pypi.org/project/rouge-score/)

In a Colab notebook, most of them are already installed, except Evaluate and rouge_score.

In [ ]:
%pip install evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.9 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=6f485047717ce53255014a90b700191f56c71e44f64afcd99884cd9fcd2f2fb4
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


We also set some configuration parameters.

Most importantly, you should select a language model to work with in this assignment and enter its HuggingFace identifier in the parameter `MODEL_NAME` below. In principle you can use any model that you want, but we recommend that you select a model that has not already been trained to follow instructions, so it should be a "pure" language model trained on raw text (similar to Assignments 1 and 2).

The selected model should be small enough to fit in your computational environment. We have verified that the 135-million parameter [`SmolLM2` model](https://huggingface.co/HuggingFaceTB/SmolLM2-135M), developed by HuggingFace, can be used to solve this assignment in a Colab notebook (free tier, T4 GPU). If you run on a cluster, you can select a larger model (and probably see more interesting results).

We also define training and test set sizes here. Again, the values below have been set so that the assignment can be solved in Colab, and you can increase these sizes to improve the quality of the fine-tuned models.

In [ ]:
%pip install langchain
%pip install langchain-community
%pip install langchain-huggingface
%pip install langchain-core
%pip install sentence_transformers
%pip install langchain-chroma

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 117.7 MB/s eta 0:00:00
   ━━━━━━━━━━━

In [ ]:
!wget https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json

--2026-06-04 15:02:59--  https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2584787 (2.5M) [text/plain]
Saving to: ‘ori_pqal.json’

ori_pqal.json       100%[===================>]   2.46M  --.-KB/s    in 0.03s   

2026-06-04 15:03:00 (85.3 MB/s) - ‘ori_pqal.json’ saved [2584787/2584787]



In [ ]:
import pandas as pd
tmp_data = pd.read_json("ori_pqal.json").T
# some labels have been defined as "maybe", only keep the yes/no answers
tmp_data = tmp_data[tmp_data.final_decision.isin(["yes", "no"])]

documents = pd.DataFrame({"abstract": tmp_data.apply(lambda row: (" ").join(row.CONTEXTS+[row.LONG_ANSWER]), axis=1),
             "year": tmp_data.YEAR})
questions = pd.DataFrame({"question": tmp_data.QUESTION,
             "year": tmp_data.YEAR,
             "gold_label": tmp_data.final_decision,
             "gold_context": tmp_data.LONG_ANSWER,
             "gold_document_id": documents.index})

In [ ]:
questions.iloc[0]

,21645374
question,Do mitochondria play a role in remodelling lac...
year,2011
gold_label,yes
gold_context,Results depicted mitochondrial dynamics in viv...
gold_document_id,21645374


# Part 1: Preprocessing

### ⚙&nbsp; Task 1.1: Loading and inspecting the dataset

The dataset [SmolTalk](https://huggingface.co/datasets/HuggingFaceTB/smoltalk) is a collection of instruction-response pairs designed for SFT of large language models for instruction following. This dataset consists of examples of user inputs with system responses.

You can load using the datasets from the HuggingFace repository as follows.

In [ ]:
from datasets import load_dataset, DatasetDict

# Load the SmolTalk dataset
smoltalk = load_dataset("HuggingFaceTB/smoltalk", "all")

README.md:   0%|          | 0.00/9.72k [00:00<?, ?B/s]

data/all/train-00000-of-00009.parquet:   0%|          | 0.00/223M [00:00<?, ?B/s]

data/all/train-00001-of-00009.parquet:   0%|          | 0.00/223M [00:00<?, ?B/s]

data/all/train-00002-of-00009.parquet:   0%|          | 0.00/223M [00:00<?, ?B/s]

data/all/train-00003-of-00009.parquet:   0%|          | 0.00/223M [00:00<?, ?B/s]

...

data/all/train-00007-of-00009.parquet:   0%|          | 0.00/224M [00:00<?, ?B/s]

data/all/train-00008-of-00009.parquet:   0%|          | 0.00/224M [00:00<?, ?B/s]

data/all/test-00000-of-00001.parquet:   0%|          | 0.00/105M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1043917 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/54948 [00:00<?, ? examples/s]

In [ ]:
import torch
SEED = 101
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MAX_TRAIN_SAMPLES = 5000
MAX_TEST_SAMPLES = 400

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"

In [ ]:
from langchain_huggingface import HuggingFacePipeline

model = HuggingFacePipeline.from_model_id(
    MODEL_NAME,
    task="text-generation",
    pipeline_kwargs={
        "max_new_tokens": 3,
        "return_full_text": False,
        "do_sample": False
    }
)
model = model.bind(stop=["\n", "1.", "2."])

out = model.invoke(questions.iloc[0].question)
print(len(out))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=3) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


0


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
out = embeddings.embed_query(questions.iloc[0].question)
print(len(out))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

384


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=256,
    length_function=len,
    is_separator_regex=False,
)

metadatas = [{"id": idx} for idx in documents.index]
texts = text_splitter.create_documents(texts=documents.abstract.tolist(), metadatas=metadatas)

In [ ]:
texts[0]

Document(metadata={'id': 21645374}, page_content='Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less')

In [ ]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    collection_metadata={"hnsw:space": "cosine"}
)

In [ ]:
from typing import Any
from langchain_core.documents import Document
from langchain.agents.middleware import AgentMiddleware, AgentState


class State(AgentState):
    context: list[Document]


class RetrieveDocumentsMiddleware(AgentMiddleware[State]):
    state_schema = State

    def __init__(self, vector_store):
        self.vector_store = vector_store

    def before_model(self, state: AgentState) -> dict[str, Any] | None:
        last_message = state["messages"][-1] # get the user input query
        retrieved_docs = self.vector_store.similarity_search(last_message.text)  # search for documents
        docs_content = ''
        #docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

        augmented_message_content = f"""Use the following context to answer the question.

        Context:
        {docs_content}

        Question:
        {last_message.text}

        Answer strictly with a single word: Yes or No.
        Answer:
        """
        return {
            "messages": [last_message.model_copy(update={"content": augmented_message_content})],
            "context": retrieved_docs,
        }

In [ ]:
vector_store.delete_collection()
vector_store = Chroma.from_documents(documents=texts, embedding=embeddings)
print(f"Number of documents in vector store: {vector_store._collection.count()}")

# Re-run the similarity search
results = vector_store.similarity_search_with_score(
    "What is programmed cell death?", k=3
)
for res, score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")

Number of documents in vector store: 5133
* [SIM=0.775410] Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less [{'id': 21645374}]
* [SIM=1.245382] ligand) was detected in all three uveal melanoma cell lines, suggesting the presence of autocrine (paracrine) stimulation pathways. Treatment of uveal melanoma cell lines with STI571, which blocks c-kit autophosphorylation, resulted in cell death. The IC(50) of the inhibitory effects on c-kit phosphorylation and cell proliferation was of equal size and less than 2.5 microM. The results confirm

In [ ]:
from langchain.agents import create_agent

rag_middleware = RetrieveDocumentsMiddleware(vector_store=vector_store)

agent = create_agent(
    model=model,
    tools=[],
    middleware=[rag_middleware]
)

In [ ]:
import re

state = agent.invoke({"messages": [{"role": "user", "content": questions.iloc[0].question}]})
raw_answer = state["messages"][-1].content
match = re.search(r'\b(yes|no)\b', raw_answer.lower())
clean_answer = match.group(1) if match else "Unknown"
print(f"Raw: {raw_answer!r} -> Clean: {clean_answer}")

Both `max_new_tokens` (=3) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Raw: '\n\nWhat' -> Clean: Unknown


With context:
Correct answers: 493/890

Without:
Correct answers: 0/890

In [ ]:
correct_answers = 0

for question in questions.itertuples():
  state = agent.invoke({"messages": [{"role": "user", "content": question.question}]})
  raw_answer = state["messages"][-1].content
  match = re.search(r'\b(yes|no)\b', raw_answer.lower())
  clean_answer = match.group(1) if match else "Unknown"
  if clean_answer == question.gold_label.lower():
    correct_answers += 1
print(f"Correct answers: {correct_answers}/{len(questions)}")


Both `max_new_tokens` (=3) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=3) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=3) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=3) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_ne

Correct answers: 0/890


In order to make this assignment possible to solve in a restricted environment, we simplify the dataset a bit:
- We remove multi-turn chat dialogues from the dataset;
- We remove instances where the query or the answer is greater than a set maximum length;
- We keep a subset of the data for training and testing (by default 5000 and 400, respectively).

In [ ]:
smoltalk_simplified = smoltalk.filter(lambda row: len(row['messages']) <= 3 and all(len(m['content']) <= 256 for m in row['messages']))
smoltalk_simplified = DatasetDict({
    "train": smoltalk_simplified["train"].select(range(MAX_TRAIN_SAMPLES)),
    "test": smoltalk_simplified["test"].select(range(MAX_TEST_SAMPLES)),
})

Filter:   0%|          | 0/1043917 [00:00<?, ? examples/s]

Filter:   0%|          | 0/54948 [00:00<?, ? examples/s]

In [ ]:
smoltalk_simplified['train'][0]

{'messages': [{'content': "You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.",
   'role': 'system'},
  {'content': 'Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.',
   'role': 'user'},
  {'content': 'The chef made more food after the restaurant ran out.',
   'role': 'assistant'}],
 'source': 'explore-instruct-rewriting'}

Print some examples from the dataset so that you understand the format.

Key points you need to note here: each example from the training or test set consists of a sequence of messages. The number of messages in each example will be 2 or 3, because we removed multi-turn chat dialogues in the previous step. Each message is associated with a `role` label:
- `user`: an example of something the user might write.
- `assistant`: an example of an output an LLM could be expected to produce, given the input.
- `system`: a *system prompt* that gives guidelines for the general behavior of the LLM's behavior.

All examples in the dataset include a user input and an assistant output, but the system prompt is not available in all of the examples.

### 🎓&nbsp; Task 1.2: Formatting the data for instruction tuning

Define a function `format_input_output` that converts an example from the dataset into an input/output pair that we can use to fine-tune the LLM.

You are free to design the format. The following document gives some examples that have been used by different instruction-following LLMs including Llama and Mistral: https://huggingface.co/learn/llm-course/chapter11/2#common-template-formats

The later stages of our preprocessing pipeline expect that this function returns an object containing two parts: the `prompt` (what goes into the LLM before generating anything) and the `response` (what the LLM is expected to generate).

In [ ]:
def format_input_output(example):
    # `messages` is a list of messages, each with a `content` string and a `role`.
    messages = example['messages']

    prompt = ""
    response = ""

    for msg in messages:
        role = msg['role']
        content = msg['content']

        if role == 'system':
            prompt += f"<|im_start|>system\n{content}<|im_end|>\n"
        elif role == 'user':
            prompt += f"<|im_start|>user\n{content}<|im_end|>\n<|im_start|>assistant\n"
        elif role == 'assistant':
            response = f"{content}<|im_end|>"

    return {"prompt": prompt, "response": response}

Apply the function you implemented to the dataset as a whole.

In [ ]:
ds_sft = smoltalk_simplified.map(format_input_output)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Then verify that the dataset now contains the new fields you created.

In [ ]:
ds_sft['train'][0]

{'messages': [{'content': "You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.",
   'role': 'system'},
  {'content': 'Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.',
   'role': 'user'},
  {'content': 'The chef made more food after the restaurant ran out.',
   'role': 'assistant'}],
 'source': 'explore-instruct-rewriting',
 'prompt': "<|im_start|>system\nYou are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.<|im_end|>\n<|im_start|>user\nRearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.<|im_end|>\n<|im_start|>assistant\n",
 'response': 'The chef made more food after the restaurant ran out.<|im_end|>'}

### ⚙&nbsp; Task 1.3: Tokenizing the dataset

We will now prepare the format required by the HuggingFace Trainer.

We first load the tokenizer for our selected model:

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

Write a function `tokenize_helper` that takes an example (using the prompt/response format from the previous step) and produces the following three results:

- `input_ids`: the integer token ids of the concatenated prompt and response;
- `labels`: a list of the same length as `input_ids`, where the response token ids are the same, but where the prompt token ids have all been replaced by the loss masking identifier -100.
- `attention_mask`: the attention mask. This should just be a list of the same length as the other two lists, with all items set to 1.

The reason why `input_ids` and `labels` are different is that
we do not want to compute the training loss for tokens that appear in the user's input. We want to train the model to generate output *conditionally*: based on a prompt. But why the magic number -100? This is the number used by default in PyTorch's [`CrossEntropyLoss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) to indicate an item that should be excluded in loss computations. (This issue was also mentioned in [Assignment 1](https://liu-nlp.ai/dl4nlp/units/a1_1.html#task-4.1-implementing-the-trainer).)

In [ ]:
def tokenize_helper(example):
    prompt = example['prompt']
    response = example['response']

    prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)
    response_ids = tokenizer.encode(response, add_special_tokens=False)

    input_ids = prompt_ids + response_ids
    attention_mask = [1] * len(input_ids)

    # mask quesiton tokens
    labels = [-100] * len(prompt_ids) + response_ids

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

As above, apply the function you implemented to the dataset using `map`. This will add the three new fields to the dataset.

In [ ]:
ds_masked = ds_sft.map(tokenize_helper)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]


## Part 2: Evaluation of the baseline model

As a first step, we will see how well the *baseline* model performs: that is, a model that has not been trained to follow instructions.

### ⚙&nbsp; Task 2.1: Preparing for evaluation

In this section, we set up a few utilities we will need to complete our training and evaluation infrastructure. These utilities will be given and you don't need to modify anything.

The first piece we need is a *collator*: that is, a tool that takes a number of instances and creates PyTorch tensors for a training batch. To make the batch fit into rectangular tensors, padding tokens will be added.

In [ ]:
def data_collator(batch):
    """
    Create a custom collate function for causal language modeling.

    Args:
        batch: List of examples, each with 'input_ids', 'attention_mask', 'labels'
        tokenizer: Tokenizer with pad_token_id
    """

    input_ids_list = [torch.tensor(example["input_ids"], dtype=torch.long) for example in batch]
    attention_masks_list = [torch.tensor(example["attention_mask"], dtype=torch.long) for example in batch]
    labels_list = [torch.tensor(example['labels'], dtype=torch.long) for example in batch]

    # Find max length in this batch
    max_len = max(x.size(0) for x in input_ids_list)

    # Helper pad function
    def pad_to_max(x_list, pad_value):
        padded = []
        for x in x_list:
            pad_len = max_len - x.size(0)
            if pad_len > 0:
                pad_tensor = torch.full((pad_len,), pad_value, dtype=x.dtype)
                x = torch.cat([x, pad_tensor], dim=0)
            padded.append(x)
        return torch.stack(padded, dim=0)

    # Use tokenizer.pad_token_id for inputs, 0 for attention_mask, -100 for labels
    pad_id = tokenizer.pad_token_id

    batch_input_ids = pad_to_max(input_ids_list, pad_value=pad_id)
    batch_attention_mask = pad_to_max(attention_masks_list, pad_value=0)
    batch_labels = pad_to_max(labels_list, pad_value=-100)

    batch = {
            "input_ids": batch_input_ids,
            "attention_mask": batch_attention_mask,
            "labels": batch_labels,
        }
    return batch

The second utility we need is an evaluator. We will use the **ROUGE-L** metric, which computes the longest common subsequence between the model's output and the gold-standard answer. You can read about ROUGE-L here: https://en.wikipedia.org/wiki/ROUGE_(metric)

When using the ROUGE-L metric in a Trainer, we need to wrap it in an object defined as follows:

In [ ]:
import evaluate

class RougeMetricComputer:
    """
    Stateful metric for batch_eval_metrics=True.

    It:
      - accumulates predictions and references across batches
      - computes ROUGE-L once at the end (compute_result=True)
    """

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.rouge = evaluate.load("rouge")
        self.all_predictions = []
        self.all_references = []

    def __call__(self, eval_pred, compute_result=False):
        """Accumulate predictions and compute at the end."""

        logits, labels = eval_pred
        pred_ids = logits.argmax(axis=-1)

        # Collect decoded answer-span text from each example in the batch
        for p, lbl in zip(pred_ids, labels):
            mask = lbl != -100
            if mask.sum() == 0:
                continue

            ref_ids = lbl[mask]
            pred_ids_filtered = p[mask]

            ref_text = self.tokenizer.decode(ref_ids, skip_special_tokens=True)
            pred_text = self.tokenizer.decode(
                pred_ids_filtered, skip_special_tokens=True,
                eos_token_id=self.tokenizer.vocab['<|im_end|>']
            )

            self.all_references.append(ref_text.strip())
            self.all_predictions.append(pred_text.strip())

        # Only compute at the very end of eval
        if compute_result:
            if len(self.all_references) > 0:
                scores = self.rouge.compute(
                    predictions=self.all_predictions,
                    references=self.all_references,
                )

                # Clear accumulated data for next eval call
                self.all_predictions = []
                self.all_references = []
                return {"rougeL": scores["rougeL"]}
            else:
                return {}
        else:
            return {}

compute_metrics = RougeMetricComputer(tokenizer)


Finally, we make a function that sets up a [`Trainer`](https://huggingface.co/docs/transformers/main_classes/trainer).

In [ ]:
from transformers import Trainer
from transformers.trainer_callback import ProgressCallback

def make_trainer(model, training_args):
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=ds_masked["train"],
        eval_dataset=ds_masked["test"],
        compute_metrics=compute_metrics,
        data_collator=data_collator,
    )
    trainer.callback_handler.callbacks = [
        cb for cb in trainer.callback_handler.callbacks
        if type(cb).__name__ != "NotebookProgressCallback"
    ]
    trainer.add_callback(ProgressCallback)
    return trainer


### 🎓&nbsp; Task 2.2: Evaluating the pre-trained model

Now, we have all the pieces to evaluate our baseline model that has not been instruction-tuned.

The following code will compute the loss on the test set as well as the ROUGE-L score. You will later compare these scores to the models that you train.

Why do you think the ROUGE-L score is as high as it is, even without any training for instruction-following?

In [ ]:
from transformers import TrainingArguments
from transformers import AutoModelForCausalLM
import time
import json

print("\n" + "=" * 80)
print("EVALUATING PRETRAINED MODEL")
print("=" * 80)

pretrained_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to("cuda")

pretrained_eval_args = TrainingArguments(
    eval_strategy="no",
    per_device_eval_batch_size=1,
    bf16=True, fp16=False,
    report_to="none",
    batch_eval_metrics=True,
    eval_accumulation_steps=1,
)

pretrained_trainer = make_trainer(pretrained_model, pretrained_eval_args)

t0 = time.perf_counter()
pretrained_eval_metrics = pretrained_trainer.evaluate()
pretrained_eval_time = time.perf_counter() - t0

pretrained_eval_loss = float(pretrained_eval_metrics["eval_loss"])
pretrained_rougeL = pretrained_eval_metrics.get("eval_rougeL", None)

print("\nPRETRAINED EVAL METRICS:")
print(json.dumps(pretrained_eval_metrics, indent=2))


EVALUATING PRETRAINED MODEL


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

  0%|          | 0/400 [00:00<?, ?it/s]


PRETRAINED EVAL METRICS:
{
  "eval_loss": 1.2489768266677856,
  "eval_model_preparation_time": 0.0196,
  "eval_rougeL": 0.6627403799988445,
  "eval_runtime": 44.7796,
  "eval_samples_per_second": 8.933,
  "eval_steps_per_second": 8.933
}



## Part 3: Supervised fine-tuning



### 🎓&nbsp; Task 3.1: Training the full model

Next, we train the pre-trained model using SFT over all the parameters, then calculate the metrics and outputs to evaluate how well it follows instructions.

How do the results differ from those in the previous step?

In [ ]:
baseline_training_args = TrainingArguments(
    eval_strategy="epoch",
    logging_steps=2000,
    save_strategy="no",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    bf16=True, fp16=False,
    report_to="none",
    batch_eval_metrics=True,
    eval_accumulation_steps=1,
)

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to("cuda")

baseline_trainer = make_trainer(base_model, baseline_training_args)

print("Starting training...")
baseline_trainer.train()

print("\nEvaluating fine-tuned baseline model...")
baseline_eval_metrics = baseline_trainer.evaluate()

print("\nBASELINE FINE-TUNED METRICS:")
print(json.dumps(baseline_eval_metrics, indent=2))

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Starting training...


  0%|          | 0/5000 [00:00<?, ?it/s]

{'loss': '1.187', 'grad_norm': '2.531', 'learning_rate': '3.001e-05', 'epoch': '0.4'}
{'loss': '1.115', 'grad_norm': '8.125', 'learning_rate': '1.001e-05', 'epoch': '0.8'}


  0%|          | 0/400 [00:00<?, ?it/s]

{'eval_loss': '1.132', 'eval_rougeL': '0.6736', 'eval_runtime': '51.65', 'eval_samples_per_second': '7.744', 'eval_steps_per_second': '7.744', 'epoch': '1'}
{'train_runtime': '835.1', 'train_samples_per_second': '5.988', 'train_steps_per_second': '5.988', 'train_loss': '1.138', 'epoch': '1'}

Evaluating fine-tuned baseline model...


  0%|          | 0/400 [00:00<?, ?it/s]


BASELINE FINE-TUNED METRICS:
{
  "eval_loss": 1.1323342323303223,
  "eval_rougeL": 0.6735607601657632,
  "eval_runtime": 35.2959,
  "eval_samples_per_second": 11.333,
  "eval_steps_per_second": 11.333,
  "epoch": 1.0
}


{
  "eval_loss": 1.2489768266677856,
  "eval_model_preparation_time": 0.0196,
  "eval_rougeL": 0.6627403799988445,
  "eval_runtime": 44.7796,
  "eval_samples_per_second": 8.933,
  "eval_steps_per_second": 8.933
}

vs

{
  "eval_loss": 1.1323342323303223,
  "eval_rougeL": 0.6735607601657632,
  "eval_runtime": 35.2959,
  "eval_samples_per_second": 11.333,
  "eval_steps_per_second": 11.333,
  "epoch": 1.0
}

### ⚙&nbsp; Task 3.3: Counting the number of trainable parameters

Define a function `num_trainable_parameters` that computes the number of floating-point numbers that a given model will update during training.

**Hints**:
- For a PyTorch module `m`, you can use `m.parameters()` to access its parameter tensors.
- However, you should only include parameter tensors where the flag `requires_grad` is True.


In [ ]:
def num_trainable_parameters(model):
    """Count number of trainable parameters.

    Args:
        model: A PyTorch module.
    """
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

trainable_params = num_trainable_parameters(pretrained_model)
print(f"Number of trainable parameters: {trainable_params:,}")
trainable_params = num_trainable_parameters(base_model)
print(f"Number of trainable parameters: {trainable_params:,}")

Number of trainable parameters: 134,515,008
Number of trainable parameters: 134,515,008


Apply this function to the SFT-trained model and check that the result makes sense.

## Part 4: Parameter-efficient fine-tuning

In the last section of this assignment, we will use LoRA to train the model in a more parameter-efficient manner. You may want to prepare by reading  by [the paper by Hu et al. (2021)](https://arxiv.org/pdf/2106.09685) and the teaching material provided for this course.

### ⚙&nbsp; Task 4.1: Utilities for modifying models

Define a function `extract_lora_targets` that extracts the relevant linear layers from all Transformer blocks in your selected LLM.
It is up to you to decide what layers to select; in the experiments described in the original LoRA paper, the query and value projection matrices were fine-tuned with LoRA, while all other layers were left unchanged.
Return a dictionary that maps the component name to the corresponding linear layer.

As we saw earlier (in Assignment 2 and elsewhere), a Transformer model consists of a hierarchy of nested submodules. Each of these can be addressed by a fully-qualified string name. You can use get_submodule() to retrieve a layer by a string name. This name depends on the model you have selected. For instance, in the `SmolLM2-135M` model, `'model.layers.0.self_attn.q_proj'`
 refers to the query projection in Transformer layer 0.

It is OK to hard-code this part, so that you just enumerate the layers you want to extract. Alternatively, use a utility such as `model.named_modules()` to iterate through the model's layers.

In [ ]:
import torch.nn as nn

def extract_lora_targets(model):
    # get the last layer
    lora_targets = {}
    target_prefix = "model.layers.29.self_attn"

    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and name.startswith(target_prefix):
            if any(suffix in name for suffix in ["q_proj", "k_proj", "v_proj", "o_proj"]):
                lora_targets[name] = module
    return lora_targets

target_layers = extract_lora_targets(base_model)
print(f"Found {len(target_layers)} target layers for LoRA (expecting 4).")
for name, _ in target_layers.items():
    print(f"Targeting: {name}")

Found 4 target layers for LoRA (expecting 4).
Targeting: model.layers.29.self_attn.q_proj
Targeting: model.layers.29.self_attn.k_proj
Targeting: model.layers.29.self_attn.v_proj
Targeting: model.layers.29.self_attn.o_proj


We also need a convenience function that puts layers back into a model. The following function does the trick. The `named_layers` argument uses the same format as returned by `extract_lora_targets`.

In [ ]:
def replace_layers(model, named_layers):
    """
    Replace submodules in `model` by name.
    """
    for name, layer in named_layers.items():
        components = name.split(".")
        submodule = model
        for comp in components[:-1]:
            submodule = getattr(submodule, comp)
        setattr(submodule, components[-1], layer)
    return model

### 🎓&nbsp; Task 4.2: Implementing the LoRA layer

To implement the LoRA approach, we define a new type of layer that will be used as a drop-in replacement for a regular linear layer.

In [the paper by Hu et al. (2021)](https://arxiv.org/pdf/2106.09685), the structure is presented visually in Figure 1, and equation (3) shows the same idea.

Start from the following skeleton and fill in the missing pieces:


In [ ]:
import torch.nn as nn
import math

class LoRALayer(nn.Module):
    def __init__(self, W, r, alpha):
        super().__init__()
        self.W = W  # The original linear layer
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        in_features = W.in_features
        out_features = W.out_features

        self.lora_A = nn.Parameter(torch.zeros((in_features, r)))
        self.lora_B = nn.Parameter(torch.zeros((r, out_features)))

        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

        self.W.weight.requires_grad = False
        if self.W.bias is not None:
            self.W.bias.requires_grad = False

    def forward(self, x):
        original_output = self.W(x)

        adapter_output = (x @ self.lora_A) @ self.lora_B

        return original_output + (adapter_output * self.scaling)

Here, `W` is the linear layer we are fine-tuning, while `r` and `alpha` are hyperparameters described in section 4.1. of the paper. The `r` parameter controls the parameter efficiency: by setting it to a low value, we save memory but make a rougher approximation. The `alpha` parameter is a scaling factor.

### 🎓&nbsp; Task 4.3: Fine-tuning with LoRA

Set up a model where you replace the four linear layers in attention blocks (query, key, value, and output) with LoRA layers. Use the following steps:
- First use `extract_lora_targets` to get the relevant linear layers.
- Each of the linear layers in the returned dictionary should be wrapped inside a LoRA layer.
- Then use `replace_layers` to put them back into the model.

Train this model and compare the training speed, metrics, and outputs to the results from Part 3.

Apply your parameter counting function (`num_trainable_parameters`) to this model, compare the results to those in Part 3, and make sure that these results correspond to your expectations.


In [ ]:
import copy
import json

lora_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to("cuda")

for param in lora_model.parameters():
    param.requires_grad = False

targets = extract_lora_targets(lora_model)

r = 8
alpha = 16
named_lora_layers = {
    name: LoRALayer(module, r=r, alpha=alpha)
    for name, module in targets.items()
}

lora_model = replace_layers(lora_model, named_lora_layers)

lora_trainable_params = num_trainable_parameters(lora_model)
print(f"Full fine-tuning trainable params: {trainable_params:,}")
print(f"LoRA fine-tuning trainable params: {lora_trainable_params:,}")
print(f"Ratio: {lora_trainable_params / trainable_params:.4%}")

lora_training_args = copy.deepcopy(baseline_training_args)
lora_trainer = make_trainer(lora_model, lora_training_args)

print("\nStarting LoRA training on the last layer...")
lora_trainer.train()

print("\nEvaluating LoRA model...")
lora_eval_metrics = lora_trainer.evaluate()
print(json.dumps(lora_eval_metrics, indent=2))

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Full fine-tuning trainable params: 134,515,008
LoRA fine-tuning trainable params: 30,720
Ratio: 0.0228%

Starting LoRA training on the last layer...


  0%|          | 0/5000 [00:00<?, ?it/s]

{'loss': '1.25', 'grad_norm': '0.1124', 'learning_rate': '3.001e-05', 'epoch': '0.4'}
{'loss': '1.195', 'grad_norm': '0.7327', 'learning_rate': '1.001e-05', 'epoch': '0.8'}


  0%|          | 0/400 [00:00<?, ?it/s]

{'eval_loss': '1.221', 'eval_rougeL': '0.6666', 'eval_runtime': '35.56', 'eval_samples_per_second': '11.25', 'eval_steps_per_second': '11.25', 'epoch': '1'}
{'train_runtime': '408', 'train_samples_per_second': '12.26', 'train_steps_per_second': '12.26', 'train_loss': '1.208', 'epoch': '1'}

Evaluating LoRA model...


  0%|          | 0/400 [00:00<?, ?it/s]

{
  "eval_loss": 1.2208300828933716,
  "eval_rougeL": 0.6666446454510047,
  "eval_runtime": 54.9353,
  "eval_samples_per_second": 7.281,
  "eval_steps_per_second": 7.281,
  "epoch": 1.0
}


### Final Comparison

| Metric | Pretrained | Full SFT | LoRA (Last Layer) |
| :--- | :--- | :--- | :--- |
| **Eval Loss** | 1.2490 | 1.1323 | 1.2208 |
| **ROUGE-L** | 0.6627 | 0.6736 | 0.6666 |
| **Trainable Params** | 0 | 134.5M | 30.7k (0.02%) |

### 🎓&nbsp; Task 4.4: Qualitative inspection

Run the three models interactively on some examples of your own choice (either taken from the training or test sets, or created by yourself). The convenience function below can be of use, but you need to complete it by using the prompt format you defined in Task 1.2.

Do your models seem to have learned the instruction-following behavior (at least to some extent)? Do they respond to user queries sensibly?

The quality we see here will depend on your choice of base model as well as how much you trained it.

In [ ]:
def run_demo(model, tokenizer, example_idx, title):
    example = ds_sft['test'][example_idx]
    # We extract the user content from the messages
    user_input = next(m['content'] for m in example['messages'] if m['role'] == 'user')

    formatted = format_input_output({'messages': [{'role': 'user', 'content': user_input}]})
    prompt = formatted['prompt']

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            eos_token_id=tokenizer.vocab['<|im_end|>'],
            do_sample=False
        )

    response = tokenizer.decode(outputs[0][len(inputs['input_ids'][0]):], skip_special_tokens=True)
    print(f"--- {title} ---\n{response.strip()}\n")

example_idx = 5
user_text = next(m['content'] for m in ds_sft['test'][example_idx]['messages'] if m['role'] == 'user')
print(f"Instruction: {user_text}\n")

run_demo(pretrained_model, tokenizer, example_idx, "PRETRAINED MODEL")
run_demo(base_model, tokenizer, example_idx, "FULL SFT MODEL")
run_demo(lora_model, tokenizer, example_idx, "LORA (LAST LAYER) MODEL")

Instruction: Replace technical terms with simpler words in this scientific publication:
The study associates the genetic variation of a specific gene with increased susceptibility to certain diseases.

--- PRETRAINED MODEL ---
The study shows that the genetic variation of a specific gene is linked to increased susceptibility to certain diseases.

--- FULL SFT MODEL ---
The study shows that the genetic variation of a specific gene can make someone more or less likely to get a disease.

--- LORA (LAST LAYER) MODEL ---
The study shows that the genetic variation of a specific gene is linked to increased susceptibility to certain diseases.

